In [1]:
T = CartanType(['A', 2])
W = WeylGroup(T,prefix="s")
[s1, s2] = W.simple_reflections()

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 7

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KL.P(y, w)(1)) * a
    return s

def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KL.P(y, w0*w)(1)) * a
    return s

def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))

def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))

def qfsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)).scale(q))

def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

Setup done, f_w, f^w are defined.


In [13]:
for w in W:
    print(w)
    print(expI(Iw(w)).weyl_group_action(w))
    print(fsub(w))
    print()

1
a2(0,0)
a2(1,1)

s1*s2
a2(-1,0)
2*a2(-1,1) + 2*a2(1,0)

s2*s1
a2(0,-1)
2*a2(1,-1) + 2*a2(0,1)

s1
a2(-1,1)
2*a2(0,1)

s2
a2(1,-1)
2*a2(1,0)

s1*s2*s1
a2(-1,-1)
6*a2(0,0)



In [3]:
woke_pairing(fsup(s1), fsup(s1)*R(rho))

-4*A2(0,1)

In [4]:
print(fsub(s1*s2))

2*a2(-1,1) + 2*a2(1,0)


In [5]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sub[i], d_sup[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sub[i], d_sup[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [6]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sup[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sub[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sub[i], d_dual[j])*d_sup[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

Computed the dual basis as the dictionary d_dual.


In [7]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    for j in range(len(W)):
        a = woke_pairing(d_sub[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

Checked dual basis, all is good.


In [8]:
#Defining [q]^*f_w
q = 11

d_qsub = {}
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_qsub[i] = qfsub(W[i])

In [9]:
#DEPENDS ON THE TYPE! Set the minimal Duflo involution in each left cell. Leave lcell_dict = {} in Type A!
# B3 EXAMPLE lcell_dict = {s1^2 : s1^2, s1 : s1, s1*s2*s3*s2*s1 : s1,
#              s2 : s2, s2*s3*s2 : s2,
##              s3 : s3, s3*s2*s3 : s3,
#              s3*s1 : s3*s1, s2*s3*s1*s2 : s2*s3*s1*s2, s3*s2*s3*s1*s2*s3 : s3*s2*s3*s1*s2*s3,
#              s1*s2*s1 : s1*s2*s1, s3*s1*s2*s3*s1 : s3*s1*s2*s3*s1, s2*s3*s1*s2*s3*s1*s2 : s2*s3*s1*s2*s3*s1*s2,
#              s3*s2*s3*s2 : s3*s2*s3*s2, s3*s2*s3*s1*s2*s3*s1*s2 : s3*s2*s3*s2,
#              s3*s1*s2*s3*s2*s1 : s3*s1*s2*s3*s2*s1, s3*s2*s3*s1*s2*s3*s2*s1 : s3*s1*s2*s3*s2*s1,
#              s1*s2*s3*s1*s2*s1 : s1*s2*s3*s1*s2*s1, s2*s3*s1*s2*s3*s1*s2*s1 : s1*s2*s3*s1*s2*s1,
#              w0 : w0}
lcell_dict = {}
for a in W:
    if a not in lcell_dict:
        lcell_dict[a] = a
print("lcell_dict is defined.")

lcell_dict is defined.


In [10]:
def nqpair(w, y):
    #new/normalized q_pairing
    return woke_pairing(d_qsub[list(W).index(w)], d_dual[list(W).index(lcell_dict[y])])

table = []

Mw = {}

for w in invols:
    table.append([str(w), str(nqpair(w0*w, w0*w))])

for row in table:
    print('| {:8} | {:1}'.format(*row))

| 1        | A2(0,0)
| s1       | A2(10,0)
| s2       | A2(0,10)
| s1*s2*s1 | A2(10,10)


In [11]:
for w in W:
    print("{:9}| {:10}".format(str(w), str(d_sup[list(W).index(w)] / 2)))

1        | 3*a2(0,0) 
s1*s2    | a2(-1,0)  
s2*s1    | a2(0,-1)  
s1       | a2(0,-1) + a2(-1,1)
s2       | a2(-1,0) + a2(1,-1)
s1*s2*s1 | 1/2*a2(-1,-1)


In [12]:
g = {}
g[s1*s1] = R(0,0)
g[s1] = R(0,1)
g[s2] = R(1,0)
g[s1*s2] = R(1,0) + R(0,2)
g[s2*s1] = R(0,1) + R(2,0)
g[s1*s2*s1] = R(1,1)
for w in W:
    print(w)
    print(fsup(s1))
    print(g[w])
    print()

1
2*a2(0,-1) + 2*a2(-1,1)
a2(0,0)

s1*s2
2*a2(0,-1) + 2*a2(-1,1)
a2(1,0) + a2(0,2)

s2*s1
2*a2(0,-1) + 2*a2(-1,1)
a2(0,1) + a2(2,0)

s1
2*a2(0,-1) + 2*a2(-1,1)
a2(0,1)

s2
2*a2(0,-1) + 2*a2(-1,1)
a2(1,0)

s1*s2*s1
2*a2(0,-1) + 2*a2(-1,1)
a2(1,1)



In [13]:
for w in W:
    for y in W:
        print(w, y, woke_pairing(fsup(w), fsup(y)))

1 1 0
1 s1*s2 0
1 s2*s1 0
1 s1 0
1 s2 0
1 s1*s2*s1 -6*A2(0,0)
s1*s2 1 0
s1*s2 s1*s2 0
s1*s2 s2*s1 -4*A2(0,0)
s1*s2 s1 0
s1*s2 s2 0
s1*s2 s1*s2*s1 -2*A2(0,1)
s2*s1 1 0
s2*s1 s1*s2 -4*A2(0,0)
s2*s1 s2*s1 0
s2*s1 s1 0
s2*s1 s2 0
s2*s1 s1*s2*s1 -2*A2(1,0)
s1 1 0
s1 s1*s2 0
s1 s2*s1 0
s1 s1 0
s1 s2 4*A2(0,0)
s1 s1*s2*s1 -2*A2(1,0)
s2 1 0
s2 s1*s2 0
s2 s2*s1 0
s2 s1 4*A2(0,0)
s2 s2 0
s2 s1*s2*s1 -2*A2(0,1)
s1*s2*s1 1 -6*A2(0,0)
s1*s2*s1 s1*s2 -2*A2(0,1)
s1*s2*s1 s2*s1 -2*A2(1,0)
s1*s2*s1 s1 -2*A2(1,0)
s1*s2*s1 s2 -2*A2(0,1)
s1*s2*s1 s1*s2*s1 -A2(1,1)


In [14]:
for w in W:
    print(w)
    print(fsup(w))
a2 = R

1
6*a2(0,0)
s1*s2
2*a2(-1,0)
s2*s1
2*a2(0,-1)
s1
2*a2(0,-1) + 2*a2(-1,1)
s2
2*a2(-1,0) + 2*a2(1,-1)
s1*s2*s1
a2(-1,-1)


In [15]:
for w in W:
    print(w, fsup(w))

1 6*a2(0,0)
s1*s2 2*a2(-1,0)
s2*s1 2*a2(0,-1)
s1 2*a2(0,-1) + 2*a2(-1,1)
s2 2*a2(-1,0) + 2*a2(1,-1)
s1*s2*s1 a2(-1,-1)
